In [11]:
### Imports ###

import requests
import math
import os

In [12]:
# Dictionary with the latitude, longitude, and the
# window width and height (in km) of the Terradets
# reservoir.

catalan_dams = {
    "escales": {
        "lat": 42.3591, 
        "lon": 0.7460,
        "width_km": 3,
        "height_km": 8
    }
}

In [13]:
# Compute the NDWI in the server and download the 
# results for all clear days (cloud_cover <= 15) 
# from January 2020 to March 2026. 

# Authentication

with open('../confidential/SH_CLIENT_ID.txt', 'r') as file:
    CLIENT_ID = file.read()

with open('../confidential/SH_CLIENT_SECRET.txt', 'r') as file:
    CLIENT_SECRET = file.read()

auth_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
res = requests.post(auth_url, data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET})
token = res.json().get('access_token')

headers = {'Authorization': f'Bearer {token}', 'Accept': 'image/tiff'}
catalog_headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Helper function that computes the window from
# the specifications in the reservoir dictionary.
# By diving by 111 (the approximate distance in 
# kilometers between two degrees of latitude), we
# transform  km in degrees.

def get_custom_bbox(lat, lon, width_km, height_km):
    lat_change = (height_km / 2) / 111.0
    lon_change = (width_km / 2) / (111.0 * math.cos(math.radians(lat)))
    return [lon - lon_change, lat - lat_change, lon + lon_change, lat + lat_change]

# Script that computes the Normalized Difference
# Water Index (NDWI). 

evalscript_ndwi = """
//VERSION=3
function setup() {
  return {
    input: ["B03", "B08", "dataMask"],
    output: { bands: 1, sampleType: "FLOAT32" } // Outputs raw math values, perfect for CNNs
  };
}
function evaluatePixel(sample) {
  if (sample.dataMask === 0) return [-2]; // -2 will represent 'No Data' padded areas
  let ndwi = (sample.B03 - sample.B08) / (sample.B03 + sample.B08);
  return [ndwi];
}
"""

# The rasters will be stored in the directory
# data/dataset_ndwi

os.makedirs("../data/escales", exist_ok=True)
catalog_url = "https://sh.dataspace.copernicus.eu/api/v1/catalog/1.0.0/search"
process_url = 'https://sh.dataspace.copernicus.eu/api/v1/process'

# Loop over the reservoirs, and each of the
# years in the period. In this case we only
# have one, but we could add more.

for dam_name, info in catalan_dams.items():
    print(f"\nProcessing {dam_name}...")
    bbox = get_custom_bbox(info['lat'], info['lon'], info['width_km'], info['height_km'])
    
    # Calculate fixed 20m/pixel resolution
    pixels_x = int((info["width_km"] * 1000) / 20)
    pixels_y = int((info["height_km"] * 1000) / 20)
    
    # We loop year-by-year to avoid Catalog API pagination limits (max 100 results per call)
    for year in range(2020, 2027):
        start_date = f"{year}-01-01T00:00:00Z"
        end_date = f"{year}-12-31T23:59:59Z"
        if year == 2026:
            end_date = "2026-03-05T23:59:59Z"
            
        print(f"  Searching Catalog for {year}...")
        
        # Step A: Ask Catalog API for valid dates
        catalog_payload = {
            "collections": ["sentinel-2-l2a"],
            "datetime": f"{start_date}/{end_date}",
            "bbox": bbox,
            "filter": "eo:cloud_cover <= 15",
            "filter-lang": "cql2-text",
            "limit": 100
        }
        
        cat_res = requests.post(catalog_url, headers=catalog_headers, json=catalog_payload)
        
        # Warning if it fails
        if cat_res.status_code != 200:
            print(f"  Catalog API Error: {cat_res.text}")
            continue
        
        cat_res = requests.post(catalog_url, headers=catalog_headers, json=catalog_payload)
        features = cat_res.json().get('features', [])
        
        # Extract unique days (sometimes Sentinel takes 2 overlapping shots on the same day)
        valid_dates = sorted(list(set([f['properties']['datetime'][:10] for f in features])))
        
        print(f"  Found {len(valid_dates)} clear days in {year}. Downloading tensors...")
        
        # Download NDWI for those exact dates
        for date_str in valid_dates:
            filepath = f"../data/escales/{dam_name}_{date_str}.tiff"
            
            # Skip if we already downloaded it (useful if script crashes and you restart)
            if os.path.exists(filepath):
                continue 
                
            process_payload = {
                "input": {
                    "bounds": {"bbox": bbox},
                    "data": [{
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date_str}T00:00:00Z", 
                                "to": f"{date_str}T23:59:59Z"
                            }
                        },
                        "type": "sentinel-2-l2a"
                    }]
                },
                "output": {
                    "width": pixels_x, "height": pixels_y,
                    "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
                },
                "evalscript": evalscript_ndwi
            }
            
            proc_res = requests.post(process_url, headers=headers, json=process_payload)
            
            if proc_res.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(proc_res.content)
            else:
                print(f"    Failed to download {date_str}: {proc_res.status_code}")

print("\nDataset generation complete!")


Processing escales...
  Searching Catalog for 2020...
  Found 35 clear days in 2020. Downloading tensors...
  Searching Catalog for 2021...
  Found 35 clear days in 2021. Downloading tensors...
  Searching Catalog for 2022...
  Found 33 clear days in 2022. Downloading tensors...
  Searching Catalog for 2023...
  Found 35 clear days in 2023. Downloading tensors...
  Searching Catalog for 2024...
  Found 39 clear days in 2024. Downloading tensors...
  Searching Catalog for 2025...
  Found 39 clear days in 2025. Downloading tensors...
  Searching Catalog for 2026...
  Found 1 clear days in 2026. Downloading tensors...

Dataset generation complete!
